In [ ]:
from dotenv import load_dotenv
import ssl
import httpx
import truststore

load_dotenv()

In [ ]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

ssl_context = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
http_client = httpx.Client(verify=ssl_context)


chat_model = init_chat_model(
    model="gpt-5.4-mini",
    http_client=http_client
)

agent = create_agent(
    model=chat_model,
    context_schema=ColourContext  
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [ ]:

print(response['messages'][-1].content)

## Accessing Context

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [ ]:
agent = create_agent(
    model=chat_model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [ ]:
from pprint import pprint
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)
print(response['messages'][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green") # Favourite colur changed. 
)

pprint(response)

In [ ]:
print(response['messages'][-1].content)